# Proyecto Oráculo

Análisis y Diseño de Algoritmos

Ustedes solo escriben en la **celda 4**. Todo lo demás ya está hecho.

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # partición de validación, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```

Cada instancia trae **hasta 5 restricciones** y se puntúa todo-o-nada: falla una, vale 0. Con pocas instancias muchas configuraciones marcan `0.0` y parecen iguales — cuántas medir es parte del problema.


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*


In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.

`cargar_modelo` acepta un alias de la tabla o cualquier id público de Hugging Face (`org/nombre`).

| Alias | Checkpoint | Tamaño | Notas |
|---|---|---|---|
| `pequeno` | `Qwen/Qwen3-1.7B` | 1.7B | el de por defecto; el más rápido |
| `ministral3b` | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.8B | fp16, ~7.7 GB de VRAM |
| `llama3b` | `unsloth/Llama-3.2-3B-Instruct` | 3.2B | fp16, ~6.4 GB de VRAM |
| `qwen8b` | `unsloth/Qwen3-8B-unsloth-bnb-4bit` | 8B | 4-bit; lento en T4 |
| `mistral7b` | `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` | 7B | 4-bit; lento en T4 |

Son tres familias distintas (Qwen, Mistral, Llama): sirve para ver si su configuración generaliza o si solo le funciona a un modelo.

El caché guarda el nombre del modelo en la clave, así que cambiar de modelo **no** reusa respuestas del anterior: vuelve a gastar rollouts.

> Usen el repo `-BF16` de Ministral 3. El repo por defecto es FP8 y la T4 no lo soporta.


In [ ]:
from ayudas import cargar_modelo

#  "pequeno"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
#  "qwen8b"      → unsloth/Qwen3-8B-unsloth-bnb-4bit
#  "mistral7b"   → unsloth/mistral-7b-instruct-v0.3-bnb-4bit
modelo = cargar_modelo("pequeno")


## 3 · El oráculo

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/oraculo_cache.json")

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Aquí escriben ustedes

Abajo hay una búsqueda aleatoria de ejemplo. Bórrenla y pongan su heurística.


In [ ]:
# ─── AQUÍ ESCRIBEN ELLOS ─────────────────────────────────────
#  EJEMPLO: búsqueda aleatoria. Bórrenlo.
# ─────────────────────────────────────────────────────────────

import random
random.seed(0)

INSTANCIAS = busqueda[:20]      # ¿con cuántas medir? ésa es su decisión
CUANTAS    = 10                 # cuántas configuraciones probar

mejor = None
historial = []
for c in random.sample(CONFIGS, CUANTAS):
    r = oraculo.evaluar(c, INSTANCIAS, semilla=1)
    historial.append(r.precision)

    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, c)

    print(f"esta {r.precision:5.1%}   mejor {mejor[0]:5.1%}")

print("\nmejor configuración:", mejor[1])

# Si casi todas marcan lo mismo (0.0 es lo típico), no es que las configs
# sean iguales: son pocas instancias para distinguirlas. Suban INSTANCIAS.


### Leer los fallos

Cada resultado trae sus trazas: mírenlas todas las veces que quieran.


In [ ]:
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida

Las 300 instancias de validación no se usaron al buscar. Sirve para ver si la config aguanta instancias nuevas, no para elegir otra.

Validarlas todas tarda; `n` escoge cuántas medir. La muestra es fija: las mismas n en cada llamada. Además imprime qué restricciones se cayeron más.


In [ ]:
# Escojan con cuántas instancias validar: más n = más confiable, pero más lento.
r_val = oraculo.validar(mejor[1], n=30)

print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")


## 5 · La entrega


In [ ]:
from ayudas import entrega
from google.colab import files

entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
